# Support vector machines

_Machine Learning_

**Find the widest possible street separating two classes.**

A support vector machine (SVM) separates two classes with a line (or plane).

     Many lines can separate the data. The SVM picks the one with the biggest gap on both sides.

---

This notebook builds up a linear SVM one step at a time. Run each cell, read the note above it, and see how the model finds the widest street between two classes and which points define its edges. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

We fit a linear SVM on a clean, well-separated dataset and read off three things the model gives us: its accuracy, the support vectors, and the weight vector. We build it in three steps.

### Step 1 — Make a separable dataset and split it

We generate 300 two-feature points in two classes with `class_sep=1.5`, which spreads the classes far enough apart that a clear gap exists. We then hold out a quarter of the points as a test set to check the boundary on unseen data.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=300, n_features=2, n_redundant=0,
                           n_clusters_per_class=1, class_sep=1.5, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)

### Step 2 — Fit the max-margin classifier

`SVC(kernel="linear")` finds the straight boundary with the widest possible margin between the two classes. The penalty `C=1.0` controls how strictly it insists every point stay on the correct side versus keeping the street wide.

In [ ]:
clf = SVC(kernel="linear", C=1.0).fit(Xtr, ytr)

### Step 3 — Inspect accuracy, support vectors, and weights

We read three results from the fitted model. Test accuracy shows how it does on unseen points. The **support vectors** are the handful of points sitting on the edge of the street — they alone define the boundary. The weight vector `w` is the direction perpendicular to that boundary.

In [ ]:
print("test accuracy:", round(clf.score(Xte, yte), 3))
print("number of support vectors:", clf.support_vectors_.shape[0])
print("weight vector w:", np.round(clf.coef_[0], 3))

## Visualize the data & results

_On two real tumor features, can a linear SVM find the widest gap between malignant and benign?_

### Step 1 — Fit an SVM on two real tumor features

We take 569 real tumors but keep just two features — mean radius and mean texture — and **standardize** them so neither dominates by scale. Fitting a linear SVM on these two features gives a boundary we can draw on a 2-D plot.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

bc = load_breast_cancer()
X = StandardScaler().fit_transform(bc.data[:, [0, 1]])   # radius, texture
y = bc.target
clf = SVC(kernel="linear", C=1.0).fit(X, y)
print("support vectors", clf.support_.size)

### Step 2 — Turn the weights into a boundary line

The SVM boundary is the line where `w0·x + w1·y + b = 0`. Solving that for the vertical axis gives `y = -(w0·x + b) / w1`, so we can sweep across the x-range and compute the matching boundary height at each point.

In [ ]:
w0, w1 = clf.coef_[0]
b = clf.intercept_[0]

# Sweep across the x-range and solve the boundary equation for the y value.
xs = np.linspace(X[:, 0].min(), X[:, 0].max(), 50)
zs = -(w0 * xs + b) / w1

### Step 3 — Plot the points and the max-margin boundary

We scatter the malignant and benign tumors, then overlay the boundary line we computed. Because the two real clouds overlap, no straight line separates them perfectly — but the SVM still places its street where the gap is widest.

In [ ]:
for label, color, name in [(0, "#ff7b72", "malignant"), (1, "#7ee787", "benign")]:
    pts = X[y == label]
    plt.scatter(pts[:, 0], pts[:, 1], c=color, label=name, edgecolor="k")
plt.plot(xs, zs, color="#ffb454", label="max-margin boundary")
plt.xlabel("mean radius (standardized)")
plt.ylabel("mean texture (standardized)")
plt.title("Linear SVM boundary")
plt.legend()
plt.show()